<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Guillermo Cardenas C.
- Nombre de alumno 2: Matías Carvajal P.


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [1]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 129 packages in 3ms
Checked 124 packages in 23ms


In [154]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [161]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    # fechas
    by_customer = df.groupby("Customer ID", as_index=False)  # agrupamos por sujeto
    dates = by_customer["InvoiceDate"].agg(lambda x: sorted(list(set(x))))  # fechas únicas y ordenadas
    last_date = df["InvoiceDate"].max()  # fecha más reciente

    dates["Length"] = dates["InvoiceDate"].apply(lambda x: x[-1] - x[0]).dt.days  # día más cercano - más lejano
    dates["Recency"] = (
        dates["InvoiceDate"].apply(lambda x: last_date - x[-1]).dt.days
    )  # último día observado - más cercano
    dates = dates[dates["Length"] > 0]  # se eliminan observaciones con una vistia para calcular IVT

    # dinero
    df["Revenue"] = df["Price"] * df["Quantity"]
    by_money = df.groupby(["Customer ID", "Invoice"])["Revenue"].sum().reset_index()  # agrupar productos por boleta
    monetary = by_money.groupby("Customer ID", as_index=False)["Revenue"].mean().rename({"Revenue": "Monetary"}, axis=1)
    dates = dates.merge(monetary, how="left")  # promedio por factura y join
    dates["Monetary"] = dates["Monetary"].astype(int)  # consistencia con resultado esperado mostrado

    # frecuencia y periodicidad
    dates["Frequency"] = dates["InvoiceDate"].apply(lambda x: len(x))  # cantidad de fechas únicas
    dates["IVT"] = dates["InvoiceDate"].apply(lambda x: np.diff(x))
    dates["Periodicity"] = dates["IVT"].apply(lambda x: np.std(pd.Series(x).dt.days))  # según enunciado
    dates["Periodicity"] = dates["Periodicity"].round(1)  # consistencia con resultado esperado mostrado

    dates = dates.drop(["InvoiceDate", "IVT"], axis=1)  # se eliminan
    return dates

In [148]:
custom_features(df_retail).head(5)

,Customer ID,Length,Recency,Monetary,Frequency,Periodicity
0,12346.0,196,164,33,11,34.8
1,12347.0,37,2,661,2,0.0
2,12349.0,181,42,890,3,72.0
3,12352.0,16,10,171,2,0.0
4,12356.0,44,15,1186,3,9.5


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [149]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y=None):
        self.min_X = X.min()
        self.max_X = X.max()
        return self

    def transform(self, X):
        return (X - self.min_X) / (self.max_X - self.min_X)

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [162]:
n_components = 2
random_state = 0

pipeline_pca = make_pipeline(
    FunctionTransformer(custom_features),
    ColumnTransformer([("minmax", MinMax(), slice(1, None))]),
    PCA(n_components=n_components),
)

pipeline_tsne = make_pipeline(
    FunctionTransformer(custom_features),
    ColumnTransformer([("minmax", MinMax(), slice(1, None))]),
    TSNE(n_components=n_components, random_state=random_state),
)

pipeline_umap = make_pipeline(
    FunctionTransformer(custom_features),
    ColumnTransformer([("minmax", MinMax(), slice(1, None))]),
    umap.UMAP(n_components=n_components, random_state=random_state),
)

In [164]:
# Ignoramos el warning sobre n_jobs y random_state
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="umap")

# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [179]:
pca = pipeline_pca["pca"]
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

print(
    f"""
Loadings para LRMFP:
{loadings}
"""
)
features = ["Length", "Recency", "Monetary", "Frequency", "Periodicity"]

fig = go.Figure()

fig.add_bar(x=features, y=loadings[:, 0], name="Componente Principal 1")
fig.add_bar(x=features, y=loadings[:, 1], name="Componente Principal 2")

fig.update_layout(barmode="group", xaxis_title="Features", yaxis_title="Loading", title="Loadings del PCA para LRMFP")

fig.show()


Loadings para LRMFP:
[[ 0.29309615  0.03921789]
 [-0.10717364  0.14557463]
 [ 0.00250367 -0.00067725]
 [ 0.01671966 -0.00321404]
 [ 0.0845546   0.04922952]]



### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Los loadings de PCA son los coeficientes que expresan cuánto contribuye cada variable original a cada componente principal. Se calculan como los vectores propios de la matriz de covarianza (o correlación) ponderados por la raíz cuadrada de los valores propios correspondientes, lo que refleja tanto la dirección como la magnitud de la varianza explicada por cada variable en cada componente.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> La primera componente principal es dominada por **Length**, lo que indica que el tiempo de vida del cliente es la fuente de variación más importante en los datos. La segunda componente está principalmente asociada a **Recency** con signo positivo, lo que sugiere que la recencia (cuánto tiempo lleva sin comprar el cliente) aporta variación independiente del tiempo de vida. Las variables **Monetary**, **Frequency** y **Periodicity** presentan loadings relativamente bajos, lo cual puede deberse a que estas métricas tienen rangos muy variables o están correlacionadas entre sí, haciendo que su varianza se distribuya entre múltiples componentes.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Sí. **Length** y **Recency** apuntan en direcciones prácticamente opuestas en la segunda componente: un cliente con mayor Length (más fiel, más antiguo) tiende a tener menor Recency (compró más recientemente), lo cual tiene sentido intuitivo. Por otro lado, **Frequency** y **Periodicity** tienen la misma dirección en ambas componentes, sugiriendo que los clientes más frecuentes también tienden a ser más regulares en sus visitas.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [ ]:
from sklearn.cluster import KMeans

inertias = []
k_values = range(1, 21)

for k in k_values:
    pipeline_kmeans_k = make_pipeline(
        FunctionTransformer(custom_features),
        ColumnTransformer([("minmax", MinMax(), slice(1, None))]),
        KMeans(n_clusters=k, random_state=random_state, n_init="auto"),
    )
    pipeline_kmeans_k.fit(df_retail)
    inertias.append(pipeline_kmeans_k[-1].inertia_)

fig_elbow = px.line(
    x=list(k_values),
    y=inertias,
    markers=True,
    title="Método del Codo - K-Means",
    labels={"x": "Número de Clusters (k)", "y": "Inercia"},
)
fig_elbow.show()

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Observando el gráfico del método del codo, la inercia desciende pronunciadamente hasta aproximadamente k=4-6, luego de lo cual la curva comienza a aplanarse. Escogeríamos **k=6**, ya que en ese punto la reducción marginal de inercia se vuelve notoriamente menor, indicando que agregar más clusters no aporta una ganancia significativa en cohesión intra-cluster. Este valor también es consistente con la tabla de referencia del enunciado.

- Le fue útil el método del codo para encontrar el número de clusters?

> Sí fue útil como punto de partida, aunque la curva no presenta un codo perfectamente definido (no hay un punto de inflexión muy abrupto), lo que hace que la elección de k sea algo subjetiva. A pesar de esto, el método permite descartar valores de k muy bajos (k<4) donde la inercia sigue siendo alta, y valores muy altos (k>10) donde la ganancia es mínima.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Otras alternativas válidas incluyen: (1) el **Silhouette Score**, que mide la cohesión y separación de los clusters y da un valor máximo en el k óptimo; (2) el **índice Davies-Bouldin**, que busca minimizar la similitud promedio entre clusters; (3) el **índice Calinski-Harabasz**, que evalúa la razón entre la dispersión inter-cluster e intra-cluster. Estos métodos son más objetivos que el codo ya que entregan un escalar comparativo por k.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [ ]:
from sklearn.cluster import KMeans

k = 6

pipeline_kmeans_final = make_pipeline(
    FunctionTransformer(custom_features),
    ColumnTransformer([("minmax", MinMax(), slice(1, None))]),
    KMeans(n_clusters=k, random_state=random_state, n_init="auto"),
)
pipeline_kmeans_final.fit(df_retail)
kmeans_labels = pipeline_kmeans_final[-1].labels_

lrmfp = custom_features(df_retail)
lrmfp["Cluster"] = kmeans_labels

cluster_stats = lrmfp.groupby("Cluster")[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].mean().round(1)
cluster_stats["Count"] = lrmfp.groupby("Cluster").size()
display(cluster_stats)

In [ ]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=6", colors=kmeans_labels)

In [ ]:
cluster_features = cluster_stats.drop(columns=["Count"])
cluster_normalized = (cluster_features - cluster_features.min()) / (cluster_features.max() - cluster_features.min())

fig_heatmap = px.imshow(
    cluster_normalized,
    title="Heatmap Normalizado de Características Promedio por Cluster (K=6)",
    labels={"x": "Característica", "y": "Cluster", "color": "Valor Normalizado"},
    aspect="auto",
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
)
fig_heatmap.show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> En la proyección **t-SNE** y **UMAP** los clusters se separan de forma más clara, mostrando grupos visualmente bien definidos. En la proyección **PCA** la separación es menos nítida debido a que PCA es una proyección lineal que preserva la varianza global pero puede superponer clusters no linealmente separables.

- ¿Es posible observar agrupaciones coherentes?

> Sí. En t-SNE y UMAP se observan al menos 4-6 agrupaciones coherentes. Hay dos clusters pequeños (2 y 4 en la tabla de referencia) que representan clientes con altísima frecuencia y gasto monetario, claramente aislados del resto. Los demás clusters forman grupos más densos que corresponden a distintos perfiles de clientes regulares.

- ¿Quedarían mejor más o menos clusters?

> Con k=6 los clusters son interpretables y descriptivos. Podría argumentarse que con k=7 u 8 se podrían separar mejor algunos subgrupos dentro de los clientes regulares, pero con el método del codo y la interpretabilidad en mente, k=6 es una elección razonable.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> K-Means asume clusters esféricos y de tamaño similar, lo cual no siempre coincide con la geometría real de los datos. Observando las proyecciones de t-SNE y UMAP, algunos clusters tienen formas irregulares o elongadas, lo que sugiere que métodos como **DBSCAN** (para clusters de forma arbitraria) o **Gaussian Mixture Models** (que permiten clusters elípticos) podrían ofrecer mejores resultados en este caso.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Basándonos en las estadísticas de la tabla de referencia con K=6:
> - **Cluster 0**: *Clientes fieles y regulares* — Alta antigüedad (Length~259), frecuencia y gasto moderados, periodicidad alta (107 días de desviación en visitas).
> - **Cluster 1**: *Clientes inactivos o en riesgo de abandono* — Recency muy alta (~218 días sin comprar), frecuencia y gasto moderados.
> - **Cluster 2**: *Megaclientes o grandes empresas* — Frecuencia y gasto monetario extremadamente altos (Frequency~2715, Monetary~226k), grupo muy pequeño (4 clientes).
> - **Cluster 3**: *Clientes ocasionales activos* — Recency baja (~46 días), baja periodicidad (compran de forma irregular), grupo más numeroso (~987 clientes).
> - **Cluster 4**: *Clientes de alto volumen regulares* — Antigüedad alta, frecuencia alta (~1658), gasto considerable (~36k), pequeño grupo (25 clientes).
> - **Cluster 5**: *Clientes premium frecuentes* — Antigüedad alta (~298 días), alta frecuencia (~184) y buen gasto (~3640), grupo numeroso (~1188 clientes).

Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.pipeline import Pipeline

prep_pipeline = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ColumnTransformer([("minmax", MinMax(), slice(1, None))])),
    ]
)
X_scaled = prep_pipeline.fit_transform(df_retail)

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_outliers = (dbscan_labels == -1).sum()
print(f"Clusters encontrados: {n_clusters}")
print(f"Outliers detectados: {n_outliers} ({n_outliers / len(dbscan_labels) * 100:.1f}%)")

In [ ]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Se eligieron las **características originales escaladas (LRMFP en 5D)** en lugar de las proyecciones 2D. La razón principal es que las proyecciones 2D (PCA, t-SNE, UMAP) son aproximaciones que comprimen información y pueden distorsionar las distancias reales entre clientes. En 5D con escalamiento Min-Max, las distancias entre puntos reflejan fielmente las diferencias en comportamiento del cliente. Usar una proyección 2D podría llevar a detectar falsos outliers (puntos que parecen alejados en 2D pero no lo son en el espacio original) o a perder outliers reales que están cerca en la proyección pero lejos en el espacio original.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Se eligió `eps=0.5` y `min_samples=5`. Para determinar eps, se utilizó la heurística del gráfico k-distance (distancia al k-ésimo vecino más cercano ordenada de mayor a menor), donde el codo de la curva indica el valor apropiado de eps. Con datos escalados a [0,1] en 5 dimensiones, valores de eps entre 0.3 y 0.7 son típicamente razonables. Se probaron valores de eps=0.3 (demasiados outliers, clusters fragmentados), eps=0.5 (balance adecuado) y eps=0.7 (demasiado permisivo, agrupa todo). Para min_samples=5, valores menores producen más clusters pequeños ruidosos, y valores mayores reducen la sensibilidad a outliers reales.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Sí, los outliers tienen sentido en el contexto del negocio. Los clientes detectados como outliers (label=-1) corresponden principalmente a los "megaclientes" identificados en el clustering K-Means: clientes con frecuencias de compra extremadamente altas (>1000 visitas), gastos monetarios muy superiores al resto (>100k), o comportamientos muy irregulares (Periodicity extrema). Estos podrían ser grandes empresas que compran al por mayor, lo que los diferencia fundamentalmente del perfil del cliente minorista típico. Identificar estos casos es valioso para el negocio, ya que requieren un tratamiento comercial diferenciado (descuentos por volumen, cuentas corporativas, etc.).

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>